In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%run "/content/drive/MyDrive/Colab Notebooks/Pawnder/colab_setup.py"

In [ ]:
import os
import sys
import glob
import json
import shutil
import re
import random
import xml.etree.ElementTree as ET
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv2D, MaxPooling2D, GlobalAveragePooling2D,
    Dense, Dropout, Flatten, Concatenate, BatchNormalization,
    TimeDistributed, LSTM, Bidirectional
)
from tensorflow.keras.applications import (
    MobileNetV2, ResNet50, EfficientNetB0
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, TensorBoard
)
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import cv2
import yaml
from collections import defaultdict
from tqdm import tqdm
from pathlib import Path
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split




try:
    import cv2
except ImportError:
    !pip install opencv-python-headless

try:
    from tqdm import tqdm
except ImportError:
    !pip install tqdm

In [ ]:
# Project paths setup
project_root = '/content/drive/MyDrive/Colab Notebooks/Pawnder'
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Try to import from config
try:
    from config import DATA_DIRS, ensure_directories, EMOTION_MAPPING
except ModuleNotFoundError:
    print("Could not import config module. Adjusting path...")
    # Try with full path
    sys.path.insert(0, os.path.abspath(project_root))
    try:
        from config import DATA_DIRS, ensure_directories, EMOTION_MAPPING
    except ModuleNotFoundError:
        # Define default configuration if import fails
        print("Creating default configuration")

        def normalize_path(path):
            """Normalize a path to avoid duplicates caused by different representations"""
            if isinstance(path, str) and path.startswith('s3://'):
                return path  # Don't normalize S3 paths
            return os.path.normpath(path)

        # Default data directories
        base_dir = "/content/drive/MyDrive/Colab Notebooks/Pawnder"
        output_dir = os.path.join(base_dir, "Data/processed")
        os.makedirs(output_dir, exist_ok=True)
        DATA_ROOT = normalize_path(os.path.join(base_dir, 'Data'))

        DATA_DIRS = {
            'raw': os.path.join(DATA_ROOT, 'Raw'),
            'stanford_original': os.path.join(DATA_ROOT, 'Raw', 'stanford_dog_pose'),
            'personal_dataset': os.path.join(DATA_ROOT, 'Raw', 'personal_dataset'),
            'matrix': os.path.join(DATA_ROOT, 'Matrix'),
            'interim': os.path.join(DATA_ROOT, 'interim'),
            'processed': os.path.join(DATA_ROOT, 'processed'),
            'models': os.path.join(base_dir, 'Models'),
        }

        # Default emotion mapping
        EMOTION_MAPPING = {
            "Happy or Playful": "Happy/Playful",
            "Relaxed": "Relaxed",
            "Submissive": "Submissive/Appeasement",
            "Excited": "Happy/Playful",
            "Drowsy or Bored": "Relaxed",
            "Curious or Confused": "Curiosity/Alertness",
            "Confident or Alert": "Curiosity/Alertness",
            "Jealous": "Stressed",
            "Stressed": "Stressed",
            "Frustrated": "Stressed",
            "Unsure or Uneasy": "Fearful/Anxious",
            "Possessive, Territorial, Dominant": "Aggressive/Threatening",
            "Fear or Aggression": "Fearful/Anxious",
            "Pain": "Stressed"
        }

        def ensure_directories():
            """Ensure all required directories exist"""
            for path in DATA_DIRS.values():
                if isinstance(path, str) and not path.startswith('s3://'):
                    Path(path).mkdir(parents=True, exist_ok=True)

            # Create subdirectories for processed data
            for split in ['train', 'validation', 'test']:
                split_path = os.path.join(DATA_DIRS['processed'], split)
                for subdir in ['images', 'annotations']:
                    path = os.path.join(split_path, subdir)
                    Path(path).mkdir(parents=True, exist_ok=True)

# Make sure directories exist
ensure_directories()

# Define configuration for the model (can be loaded from YAML if available)
def load_config(config_path="config.yaml"):
    """Load configuration from YAML file or use defaults"""
    if os.path.exists(config_path):
        with open(config_path, 'r') as f:
            config = yaml.safe_load(f)
        return config
    else:
        # Use default configuration
        print(f"Config file not found: {config_path}, using defaults")
        default_config = {
            "data": {
                "base_dir": project_root,
                "raw_data_dir": os.path.join(DATA_DIRS['raw']),
                "processed_data_dir": os.path.join(DATA_DIRS['processed']),
                "train_split": 0.7,
                "val_split": 0.15,
                "test_split": 0.15
            },
            "model": {
                "image_size": [224, 224, 3],
                "batch_size": 32,
                "learning_rate": 0.001,
                "dropout_rate": 0.5,
                "backbone": "mobilenetv2",
                "early_stopping_patience": 5
            },
            "training": {
                "checkpoint_dir": os.path.join(DATA_DIRS['models'], "checkpoints"),
                "logs_dir": os.path.join(DATA_DIRS['models'], "logs")
            },
            "inference": {
                "confidence_threshold": 0.6,
                "behavior_threshold": 0.5,
                "output_dir": os.path.join(DATA_DIRS['processed'], "predictions")
            }
        }

        # Save default config for future use
        os.makedirs(os.path.dirname(config_path), exist_ok=True)
        with open(config_path, 'w') as f:
            yaml.dump(default_config, f)

        return default_config

def load_config(config_path="/content/drive/MyDrive/Colab Notebooks/Pawnder/config.yaml"):
    try:
        with open(config_path, 'r') as f:
            config = yaml.safe_load(f)
        print(f"Successfully loaded config from {config_path}")
        return config
    except FileNotFoundError:
        print(f"Config file not found at: {config_path}")
        # Use the default config creation logic from the previous function
        return create_default_config(config_path)
    except PermissionError:
        print(f"Permission denied when trying to access: {config_path}")
        return create_default_config(config_path)
    except Exception as e:
        print(f"An error occurred when loading config: {str(e)}")
        return create_default_config(config_path)

# Define emotion classes from the EMOTION_MAPPING
CLASS_NAMES = list(set(EMOTION_MAPPING.values()))
# Safe class names for directories
SAFE_CLASS_NAMES = [emotion.replace("/", "_").replace("\\", "_") for emotion in CLASS_NAMES]
CLASS_MAP = dict(zip(CLASS_NAMES, SAFE_CLASS_NAMES))

In [ ]:
%run "/content/drive/MyDrive/Colab Notebooks/Pawnder/behavior_matrix_processor.py"

In [ ]:
%run "/content/drive/MyDrive/Colab Notebooks/Pawnder/dog_emotion_demo.py" --train --epochs 50 --batch_size 32